In [5]:
# ============================================================
# 1. Import libraries
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 2. Load dataset
# ============================================================

# Adjust the path/filename as needed
df = pd.read_csv("winequality-red.csv", sep=";")
print("Dataset loaded successfully.")
print(df.head())


# ============================================================
# 3. Handle missing values
# ============================================================

df = df.fillna(df.median(numeric_only=True))


# ============================================================
# 4. Map quality to 3 classes (Low, Medium, High)
# ============================================================

def map_quality(q):
    if q <= 4:
        return 0   # Low
    elif q <= 6:
        return 1   # Medium
    else:
        return 2   # High

df["quality_class"] = df["quality"].apply(map_quality)


# ============================================================
# 5. Train/test split
# ============================================================

X = df.drop(["quality", "quality_class"], axis=1)
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# ============================================================
# 6. Standardize features (for Logistic Regression)
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 7. Logistic Regression
# ============================================================

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

log_acc = accuracy_score(y_test, log_pred)
log_f1 = f1_score(y_test, log_pred, average="weighted")

print("\n===== Logistic Regression =====")
print("Accuracy:", log_acc)
print("F1 Score:", log_f1)


# ============================================================
# 8. Baseline Random Forest
# ============================================================

rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average="weighted")

print("\n===== Random Forest (Baseline) =====")
print("Accuracy:", rf_acc)
print("F1 Score:", rf_f1)


# ============================================================
# 9. AutoML with RandomizedSearchCV (Random Forest tuning)
# ============================================================

param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

rf_base = RandomForestClassifier(random_state=42)

search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=25,                 # number of random combinations
    scoring="f1_weighted",
    cv=3,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
auto_pred = best_model.predict(X_test)

auto_acc = accuracy_score(y_test, auto_pred)
auto_f1 = f1_score(y_test, auto_pred, average="weighted")

print("\n===== AutoML (RandomizedSearchCV on Random Forest) =====")
print("Best Params:", search.best_params_)
print("Accuracy:", auto_acc)
print("F1 Score:", auto_f1)


# ============================================================
# 10. Final comparison
# ============================================================

print("\n===== MODEL COMPARISON =====")
print(f"Logistic Regression:        Acc={log_acc:.4f}, F1={log_f1:.4f}")
print(f"Random Forest (Baseline):   Acc={rf_acc:.4f}, F1={rf_f1:.4f}")
print(f"AutoML (RandomizedSearch):  Acc={auto_acc:.4f}, F1={auto_f1:.4f}")

Dataset loaded successfully.
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8   